# Zazo Brain — دفتر التدريب والنشر الموحد

هذا هو دفتر المشروع الأساسي. يقرأ أحدث تصدير معتمد من Cloudflare KV عبر `latest.json` و`readyChunkKeys` فقط، ويدرب Gemma 2B LoRA، ويحفظ إصدارًا مستقلًا في Google Drive، ثم يرفع المحوّل مباشرةً إلى Cloudflare Workers AI ويحدّث مؤشر الإصدار النشط في KV.

لا تُحفظ أي رموز داخل ملف الدفتر أو GitHub. أضفها مرة واحدة في لوحة **Colab → Secrets**. الرفع المباشر من Colab يتجاوز حد Worker البالغ 80 MiB؛ حد Cloudflare AI للمحوّل أقل من 300 MB. إذا زاد الملف عن 80 MiB يحاول الدفتر إنشاء نسخة رفع FP16 مؤقتة مع إبقاء النسخة الأصلية المؤرشفة في Drive.

**مهم:** قبل التدريب، شغّل GitHub Action `rag-dataset-prep` إذا أضفت بيانات جديدة ولم يظهر وقت تصدير حديث. يوقف الدفتر عند غياب الـmanifest، أو عدم تطابق عدد السجلات، أو ثبات بصمة البيانات دون تفعيل إعادة التدريب اليدوية.

## 1. تثبيت الحزم

In [ ]:
# ثبّت الحزم اللازمة للتدريب والرفع المتدفق المباشر.
%pip install -q -U transformers datasets peft trl accelerate huggingface_hub safetensors requests-toolbelt

## 2. تحقق من GPU

In [ ]:
import torch, sys
print("Python:", sys.version)
print("PyTorch:", torch.__version__)
print("CUDA متاحة:", torch.cuda.is_available())
print("عدد وحدات GPU:", torch.cuda.device_count())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
else:
    print("لا تتابع إلى تدريب النموذج حتى تتصل جلسة Colab بوحدة GPU.")


## 3. إعداد التوكنات والحفظ الدائم

يمكنك إضافة `CF_KV_TOKEN` و`CF_AI_TOKEN` و`HF_TOKEN` إلى Colab Secrets، أو ترك الخلية التالية تطلبها مرة واحدة وتحفظها مشفرة داخل Google Drive. لا تظهر القيم أثناء الإدخال. إذا استخدمت الحفظ المشفر، احتفظ بكلمة المرور خارج الدفتر؛ ستحتاجها عند فتح Runtime جديد.


In [ ]:
import os, json, time, urllib.parse, requests, hashlib, base64, getpass, secrets
from pathlib import Path
from datetime import datetime, timezone
from google.colab import drive, userdata

CF_ACCOUNT_ID = "f116fb028284f528f1eeb6f487088acf"
ARCHIVE_KV_NAMESPACE_ID = "95f488c06b3349d9aa57f3c2ec4ff20a"
ACTIVE_CONTEXT_KV_NAMESPACE_ID = "3131f2d053e34dc3b81306822592cf32"
DRIVE_ROOT = Path("/content/drive/MyDrive/zazo_brain_lora")
drive.mount("/content/drive", force_remount=False)
DRIVE_ROOT.mkdir(parents=True, exist_ok=True)
TOKEN_STORE = DRIVE_ROOT / ".zazo_tokens.enc.json"

# الأسرار لا تُحفظ نصًا واضحًا. عند أول تشغيل يطلب الدفتر التوكنات وكلمة مرور تشفير
# ويضع ملفًا مشفرًا داخل مجلد Zazo في Drive. كلمة المرور لا تُحفظ؛ احتفظ بها لديك.
def _read_colab_secret(name):
    try:
        value = userdata.get(name)
        return value.strip() if value else None
    except Exception:
        return None

def _derive_key(password, salt):
    from cryptography.hazmat.primitives import hashes
    from cryptography.hazmat.primitives.kdf.pbkdf2 import PBKDF2HMAC
    return base64.urlsafe_b64encode(PBKDF2HMAC(
        algorithm=hashes.SHA256(), length=32, salt=salt, iterations=390000,
    ).derive(password.encode("utf-8")))

def _decrypt_store(password):
    from cryptography.fernet import Fernet
    payload = json.loads(TOKEN_STORE.read_text(encoding="utf-8"))
    key = _derive_key(password, base64.urlsafe_b64decode(payload["salt"]))
    values = json.loads(Fernet(key).decrypt(payload["ciphertext"].encode("ascii")).decode("utf-8"))
    if not values.get("CF_KV_TOKEN") or not values.get("CF_AI_TOKEN"):
        raise RuntimeError("ملف التوكنات المشفر ناقص CF_KV_TOKEN أو CF_AI_TOKEN.")
    return values

def _create_store(values):
    from cryptography.fernet import Fernet
    print("أنشئ كلمة مرور لتشفير التوكنات. لن يحفظها الدفتر؛ إذا نسيتها ستحتاج إدخال التوكنات من جديد.")
    password = getpass.getpass("كلمة مرور تشفير توكنات Zazo: ")
    if len(password) < 8:
        raise RuntimeError("استخدم كلمة مرور من 8 أحرف على الأقل.")
    salt = secrets.token_bytes(16)
    key = _derive_key(password, salt)
    encrypted = Fernet(key).encrypt(json.dumps(values, ensure_ascii=False).encode("utf-8")).decode("ascii")
    TOKEN_STORE.write_text(json.dumps({"version": 1, "salt": base64.urlsafe_b64encode(salt).decode("ascii"), "ciphertext": encrypted}, indent=2), encoding="utf-8")
    print("تم حفظ التوكنات مشفرة في:", TOKEN_STORE)

def load_zazo_tokens():
    values = {name: _read_colab_secret(name) for name in ("CF_KV_TOKEN", "CF_AI_TOKEN", "HF_TOKEN")}
    required_missing = not values["CF_KV_TOKEN"] or not values["CF_AI_TOKEN"]
    if required_missing and TOKEN_STORE.exists():
        try:
            values.update(_decrypt_store(getpass.getpass("كلمة مرور توكنات Zazo المحفوظة: ")))
            print("تم فك تشفير توكنات Zazo من Google Drive داخل الذاكرة فقط.")
        except Exception as error:
            raise RuntimeError("تعذر فك ملف التوكنات. تحقق من كلمة المرور أو احذف الملف المشفر وأعد الإدخال.") from error
    if not values["CF_KV_TOKEN"]:
        values["CF_KV_TOKEN"] = getpass.getpass("CF_KV_TOKEN (Cloudflare KV): ").strip()
    if not values["CF_AI_TOKEN"]:
        values["CF_AI_TOKEN"] = getpass.getpass("CF_AI_TOKEN (Cloudflare Workers AI): ").strip()
    if not values["HF_TOKEN"]:
        values["HF_TOKEN"] = getpass.getpass("HF_TOKEN (اختياري؛ اضغط Enter للتخطي): ").strip() or None
    if not values["CF_KV_TOKEN"] or not values["CF_AI_TOKEN"]:
        raise RuntimeError("CF_KV_TOKEN وCF_AI_TOKEN مطلوبان.")
    if not TOKEN_STORE.exists() and not all(_read_colab_secret(name) for name in ("CF_KV_TOKEN", "CF_AI_TOKEN")):
        _create_store(values)
    return values

_tokens = load_zazo_tokens()
CF_KV_TOKEN = _tokens["CF_KV_TOKEN"]
CF_AI_TOKEN = _tokens["CF_AI_TOKEN"]
HF_TOKEN = _tokens.get("HF_TOKEN")
MODEL_NAME = "google/gemma-2b-it"
CF_AI_MODEL = "@cf/google/gemma-2b-it-lora"
ACTIVE_LORA_KEY = "active-lora"
LATEST_EXPORT_KEY = "exports/zazo-rag-prepared/latest.json"
MAX_EXPORT_AGE_HOURS = 72
ALLOW_STALE_EXPORT = False
FORCE_RETRAIN_SAME_DATA = False
WARN_AT_ADAPTERS = 80
STOP_AT_ADAPTERS = 95
KEEP_RECENT_ADAPTERS = 5
LORA_R = 8
LORA_TARGET_MODULES = ["q_proj", "k_proj", "v_proj", "o_proj"]
MAX_SEQ_LENGTH = 1024
COMPUTE_DTYPE_NAME = "float16"

def sha256_file(file_path):
    digest = hashlib.sha256()
    with open(file_path, "rb") as stream:
        for block in iter(lambda: stream.read(1024 * 1024), b""):
            digest.update(block)
    return digest.hexdigest()

print("تم تحميل الإعدادات والتوكنات داخل الذاكرة فقط.")
print("مجلد الإصدارات:", DRIVE_ROOT / "versions")


## 4. أدوات Cloudflare API وقراءة KV

تُستخدم namespace الأرشيف للقراءة فقط، بينما يكتب الدفتر مفتاح `active-lora` في namespace السياق بعد نجاح رفع الأوزان. لا تتداخل هذه القيمة مع المفتاح الحالي `active-context`.

In [ ]:
CF_API_BASE = f"https://api.cloudflare.com/client/v4/accounts/{CF_ACCOUNT_ID}"
KV_API_BASE = CF_API_BASE + "/storage/kv/namespaces"
AI_API_BASE = CF_API_BASE + "/ai"


def kv_value_url(namespace_id, key):
    return f"{KV_API_BASE}/{namespace_id}/values/{urllib.parse.quote(key, safe='')}"


def read_kv_text(namespace_id, key, token=CF_KV_TOKEN):
    response = requests.get(
        kv_value_url(namespace_id, key),
        headers={"Authorization": f"Bearer {token}"},
        timeout=90,
    )
    if response.status_code == 404:
        return None
    if not response.ok:
        raise RuntimeError(f"تعذرت قراءة KV key={key}: HTTP {response.status_code}: {response.text[:500]}")
    return response.text


def write_kv_json(namespace_id, key, value, token=CF_KV_TOKEN):
    body = json.dumps(value, ensure_ascii=False, separators=(",", ":")).encode("utf-8")
    response = requests.put(
        kv_value_url(namespace_id, key),
        headers={
            "Authorization": f"Bearer {token}",
            "Content-Type": "application/json; charset=utf-8",
        },
        data=body,
        timeout=90,
    )
    if not response.ok:
        raise RuntimeError(f"تعذرت كتابة KV key={key}: HTTP {response.status_code}: {response.text[:500]}")
    payload = response.json()
    if not payload.get("success", False):
        raise RuntimeError(f"Cloudflare KV رفض كتابة {key}: {payload.get('errors', [])}")
    return payload


def cloudflare_json(method, url, token, **kwargs):
    response = requests.request(
        method,
        url,
        headers={"Authorization": f"Bearer {token}"},
        timeout=(30, 180),
        **kwargs,
    )
    try:
        payload = response.json()
    except ValueError as exc:
        raise RuntimeError(f"Cloudflare أعاد استجابة غير JSON: HTTP {response.status_code}") from exc
    if not response.ok or payload.get("success") is False:
        raise RuntimeError(f"Cloudflare API HTTP {response.status_code}: {payload.get('errors', response.text[:500])}")
    return payload


def flatten_result_items(result):
    if isinstance(result, dict):
        return [result]
    if isinstance(result, list):
        return [item for child in result for item in flatten_result_items(child)]
    return []

## 3. إعداد التوكنات والحفظ الدائم

يمكنك إضافة `CF_KV_TOKEN` و`CF_AI_TOKEN` و`HF_TOKEN` إلى Colab Secrets، أو ترك الدفتر يطلبها مرة واحدة ويحفظها مشفرة داخل Google Drive. لا تحفظ كلمة مرور التشفير في الدفتر؛ ستحتاجها عند فتح Runtime جديد إذا لم تستخدم Colab Secrets.


In [ ]:
from pathlib import Path
import csv

INPUT_DIR = DRIVE_ROOT / "input"
INPUT_DIR.mkdir(parents=True, exist_ok=True)
TRAINING_INPUT_NAME = os.getenv("ZAZO_TRAINING_INPUT", "").strip()
SUPPORTED_INPUT_SUFFIXES = {".jsonl", ".ndjson", ".json", ".csv", ".tsv"}

def find_training_input():
    if TRAINING_INPUT_NAME:
        selected = INPUT_DIR / TRAINING_INPUT_NAME
        if not selected.exists():
            raise RuntimeError(f"ملف التدريب المحدد غير موجود: {selected}")
        return selected
    preferred = [INPUT_DIR / name for name in ("training.jsonl", "training.ndjson", "training.json", "training.csv", "training.tsv")]
    for candidate in preferred:
        if candidate.exists():
            return candidate
    candidates = [path for path in INPUT_DIR.iterdir() if path.is_file() and path.suffix.lower() in SUPPORTED_INPUT_SUFFIXES]
    if not candidates:
        print(f"لم يوجد ملف تدريب في {INPUT_DIR}. اختر الملف من جهازك الآن أو أوقف الخلية وضعه في مجلد input.")
        try:
            from google.colab import files
            uploaded = files.upload()
        except Exception as error:
            raise RuntimeError(f"ضع ملف التدريب في Google Drive داخل {INPUT_DIR}. الصيغ المدعومة: JSONL, NDJSON, JSON, CSV, TSV.") from error
        for name, content in uploaded.items():
            target = INPUT_DIR / Path(name).name
            if target.suffix.lower() not in SUPPORTED_INPUT_SUFFIXES:
                raise RuntimeError(f"صيغة غير مدعومة: {target.name}. استخدم JSONL أو NDJSON أو JSON أو CSV أو TSV.")
            target.write_bytes(content)
        candidates = [path for path in INPUT_DIR.iterdir() if path.is_file() and path.suffix.lower() in SUPPORTED_INPUT_SUFFIXES]
    if not candidates:
        raise RuntimeError(f"لم يتم اختيار ملف. ضع ملف التدريب في Google Drive داخل {INPUT_DIR}.")
    return max(candidates, key=lambda path: path.stat().st_mtime)

def load_drive_records(path):
    text = path.read_text(encoding="utf-8-sig")
    suffix = path.suffix.lower()
    if suffix in {".jsonl", ".ndjson"}:
        records = [json.loads(line) for line in text.splitlines() if line.strip()]
    elif suffix == ".json":
        payload = json.loads(text)
        records = payload if isinstance(payload, list) else payload.get("records", payload.get("data", [])) if isinstance(payload, dict) else []
    else:
        delimiter = "\t" if suffix == ".tsv" else ","
        records = list(csv.DictReader(text.splitlines(), delimiter=delimiter))
    if not isinstance(records, list) or not records:
        raise RuntimeError(f"ملف التدريب فارغ أو لا يحتوي قائمة سجلات: {path.name}")
    if not all(isinstance(item, dict) for item in records):
        raise RuntimeError("يجب أن يكون كل سجل في ملف التدريب JSON object أو صفًا في CSV/TSV.")
    return records

training_input_path = find_training_input()
raw_records = load_drive_records(training_input_path)
dataset_sha256 = hashlib.sha256(training_input_path.read_bytes()).hexdigest()
manifest_key = f"drive://{training_input_path.relative_to(DRIVE_ROOT).as_posix()}"
created_at = datetime.fromtimestamp(training_input_path.stat().st_mtime, timezone.utc).isoformat()
expected_ready_rows = len(raw_records)
# توافق مع خلايا التدريب القديمة: ملف Drive لا يحتوي reviewRows من manifest Cloudflare.
manifest = {"reviewRows": 0}

print("مصدر التدريب:", training_input_path)
print("عدد السجلات:", len(raw_records))
print("بصمة dataset SHA-256:", dataset_sha256)

latest_training_path = DRIVE_ROOT / "latest_training.json"
if latest_training_path.exists():
    previous_training = json.loads(latest_training_path.read_text(encoding="utf-8"))
    previous_hash = previous_training.get("dataset_sha256")
    if previous_hash == dataset_sha256 and not FORCE_RETRAIN_SAME_DATA:
        raise RuntimeError("هذه هي نفس بصمة بيانات آخر تدريب ناجح؛ غيّر الملف أو اضبط FORCE_RETRAIN_SAME_DATA=True لإعادة التدريب عمدًا.")
    if previous_hash:
        print("البصمة السابقة:", previous_hash, "— هل تغير المحتوى؟", previous_hash != dataset_sha256)
print("اكتمل فحص سلامة مصدر التدريب من Google Drive.")


## 6. تحويل السجلات إلى محادثات تدريب

In [ ]:
def normalize_record(item):
    # صيغة محادثات جاهزة messages
    messages = item.get("messages")
    if isinstance(messages, list) and messages:
        cleaned = []
        for msg in messages:
            if not isinstance(msg, dict):
                continue
            role = msg.get("role")
            content = msg.get("content")
            if role in {"system", "user", "assistant"} and isinstance(content, str) and content.strip():
                cleaned.append({"role": role, "content": content.strip()})
        if any(m["role"] == "assistant" for m in cleaned):
            return {"messages": cleaned}

    # صيغة Zazo JSONL: instruction/context/response
    instruction = item.get("instruction") or item.get("question") or item.get("prompt") or item.get("input")
    response = item.get("response") or item.get("answer") or item.get("output")
    context = item.get("context") or item.get("system") or ""
    if isinstance(instruction, str) and instruction.strip() and isinstance(response, str) and response.strip():
        user_content = instruction.strip()
        if isinstance(context, str) and context.strip():
            user_content += "\n\nالسياق:\n" + context.strip()
        return {"messages": [
            {"role": "user", "content": user_content},
            {"role": "assistant", "content": response.strip()},
        ]}
    return None

examples = []
rejected = 0
for record in raw_records:
    normalized = normalize_record(record)
    if normalized:
        examples.append(normalized)
    else:
        rejected += 1

print("أمثلة التدريب الصالحة:", len(examples))
print("سجلات متجاوزة لغياب السؤال/الإجابة:", rejected)
if not examples:
    raise RuntimeError(
        "لم يتم العثور على أمثلة صالحة ضمن readyChunkKeys المعلنة في manifest. "
        f"السجلات الخام المقروءة: {len(raw_records)}؛ راجع صيغة messages في التصدير."
    )
print("حقول أول مثال:", list(examples[0].keys()))
print("عدد الرسائل في أول مثال:", len(examples[0]["messages"]))


## 7. Hugging Face وتحميل Gemma

إذا كان Gemma gated، اقبل الشروط على Hugging Face وأضف `HF_TOKEN` إلى Colab Secrets. الرمز اختياري فقط إذا كان النموذج متاحًا لحسابك دون تسجيل دخول.

In [ ]:
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM
from peft import LoraConfig, get_peft_model
from huggingface_hub import login

if not torch.cuda.is_available():
    raise RuntimeError("لا توجد GPU متاحة؛ يلزم GPU لتدريب هذا النموذج. CPU/TPU لا يعملان مع إعداد هذه الخلية.")

if HF_TOKEN:
    login(token=HF_TOKEN, add_to_git_credential=False)
compute_dtype = torch.float16

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME, token=HF_TOKEN or None)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token
model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    torch_dtype=compute_dtype,
    device_map="auto",
    token=HF_TOKEN or None,
)
model.config.use_cache = False
print("تم تحميل النموذج غير المكمّم على:", next(model.parameters()).device)

## 8. إضافة LoRA وتجهيز نصوص التدريب

In [ ]:
from datasets import Dataset
lora_config = LoraConfig(
    r=LORA_R,
    lora_alpha=LORA_R * 2,
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM",
    target_modules=LORA_TARGET_MODULES,
)
model = get_peft_model(model, lora_config)
model.print_trainable_parameters()

def render_messages(example):
    rendered = tokenizer.apply_chat_template(
        example["messages"],
        tokenize=False,
        add_generation_prompt=False,
    )
    return {"text": rendered}

train_dataset = Dataset.from_list(examples).map(render_messages)
print("عدد أمثلة التدريب:", len(train_dataset))
print("معاينة أول مثال:\n", train_dataset[0]["text"][:1200])

## 9. التدريب باستخدام TRL SFTTrainer

In [ ]:
from trl import SFTConfig, SFTTrainer

TRAIN_OUTPUT = "/content/zazo_brain_checkpoints"
training_args = SFTConfig(
    output_dir=TRAIN_OUTPUT,
    num_train_epochs=2,
    per_device_train_batch_size=1,
    gradient_accumulation_steps=8,
    learning_rate=2e-4,
    warmup_ratio=0.03,
    logging_steps=5,
    save_strategy="epoch",
    fp16=(compute_dtype == torch.float16),
    bf16=(compute_dtype == torch.bfloat16),
    gradient_checkpointing=True,
    max_length=MAX_SEQ_LENGTH,
    dataset_text_field="text",
    packing=False,
    report_to="none",
    optim="adamw_torch",
)

trainer = SFTTrainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    processing_class=tokenizer,
)
print("بدء التدريب…")
train_result = trainer.train()
print("اكتمل التدريب.")
print(train_result)


## 10. حفظ إصدار مستقل من محوّل LoRA في Google Drive

يحفظ كل تشغيل في مجلد إصدار منفصل مع بصمة البيانات ومعرّف manifest. لا يكتب فوق الإصدار السابق؛ هذا يسهّل الرجوع والأرشفة.

In [ ]:
from pathlib import Path

RUN_TIMESTAMP = datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%SZ")
RUN_ID = f"{RUN_TIMESTAMP}-sha{dataset_sha256[:10]}-r{LORA_R}"
DRIVE_VERSION_DIR = DRIVE_ROOT / "versions" / RUN_ID
DRIVE_VERSION_DIR.mkdir(parents=True, exist_ok=False)

model.save_pretrained(str(DRIVE_VERSION_DIR), safe_serialization=True)
tokenizer.save_pretrained(str(DRIVE_VERSION_DIR))

adapter_config_path = DRIVE_VERSION_DIR / "adapter_config.json"
adapter_weights_path = DRIVE_VERSION_DIR / "adapter_model.safetensors"
if not adapter_config_path.exists() or not adapter_weights_path.exists():
    raise RuntimeError("لم ينتج التدريب adapter_config.json وadapter_model.safetensors كما هو مطلوب.")

adapter_config = json.loads(adapter_config_path.read_text(encoding="utf-8"))
adapter_config["model_type"] = "gemma"
adapter_config_path.write_text(json.dumps(adapter_config, ensure_ascii=False, indent=2), encoding="utf-8")

training_metadata = {
    "run_id": RUN_ID,
    "status": "trained_not_uploaded",
    "source_manifest_key": manifest_key,
    "source_manifest_created_at": created_at,
    "dataset_sha256": dataset_sha256,
    "ready_rows": len(raw_records),
    "review_rows_excluded": manifest.get("reviewRows", 0),
    "lora_rank": LORA_R,
    "target_modules": LORA_TARGET_MODULES,
    "adapter_weights_bytes": adapter_weights_path.stat().st_size,
    "adapter_weights_sha256": sha256_file(adapter_weights_path),
    "created_at": datetime.now(timezone.utc).isoformat(),
}
(DRIVE_VERSION_DIR / "training_manifest.json").write_text(
    json.dumps(training_metadata, ensure_ascii=False, indent=2), encoding="utf-8"
)
print("حُفظ الإصدار الأصلي في:", DRIVE_VERSION_DIR)
print(f"حجم الأوزان الأصلية: {adapter_weights_path.stat().st_size / (1024**2):.1f} MiB")

## 21. تجهيز نسخة Android محلية

يحوّل هذا القسم Gemma 2B مع LoRA إلى ملفي FlatBuffer يدعمهما MediaPipe LLM Inference على Android: نموذج أساسي وملف LoRA. هذا هو المسار المدعوم لـ Gemma 2B المخصّص، بينما LiteRT-LM يستخدم نماذج `.litertlm` الجاهزة. يتطلب التحويل GPU ومساحة Drive كافية، ويُنتج حزمة `zazo_android_model.zip`.


In [ ]:
# Google MediaPipe converter: Gemma 2B + LoRA -> base.bin + lora.bin
# التشغيل اختياري لكنه مفعّل افتراضيًا بعد نجاح التدريب.
import os, shutil, subprocess, sys, zipfile
from huggingface_hub import snapshot_download

BUILD_ANDROID_MODEL = True
if BUILD_ANDROID_MODEL:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "mediapipe"])
    import mediapipe as mp
    from mediapipe.tasks.python.genai import converter

    ANDROID_DIR = DRIVE_VERSION_DIR / "android"
    ANDROID_DIR.mkdir(parents=True, exist_ok=True)
    BASE_MODEL_DIR = DRIVE_ROOT / "_cache" / MODEL_NAME.replace("/", "_")
    BASE_MODEL_DIR.parent.mkdir(parents=True, exist_ok=True)
    snapshot_download(
        repo_id=MODEL_NAME,
        local_dir=str(BASE_MODEL_DIR),
        token=HF_TOKEN or None,
        allow_patterns=["*.json", "*.safetensors", "*.model", "*.txt"],
    )

    base_output = ANDROID_DIR / "zazo_base_gpu.bin"
    lora_output = ANDROID_DIR / "zazo_lora_gpu.bin"
    intermediate = ANDROID_DIR / "intermediate"
    conversion_config = converter.ConversionConfig(
        input_ckpt=str(BASE_MODEL_DIR),
        ckpt_format="safetensors",
        model_type="GEMMA_2B",
        backend="gpu",
        output_dir=str(intermediate),
        combine_file_only=False,
        vocab_model_file=str(BASE_MODEL_DIR),
        output_tflite_file=str(base_output),
        lora_ckpt=str(adapter_weights_path),
        lora_rank=LORA_R,
        lora_output_tflite_file=str(lora_output),
    )
    converter.convert_checkpoint(conversion_config)
    if not base_output.exists() or not lora_output.exists():
        raise RuntimeError("لم ينتج المحول ملفات Android الأساسية وLoRA")

    bundle_manifest = {
        "format": "mediapipe-llm-inference-gemma-2b-lora",
        "model_name": MODEL_NAME,
        "backend": "gpu",
        "base_file": base_output.name,
        "lora_file": lora_output.name,
        "lora_rank": LORA_R,
        "target_modules": LORA_TARGET_MODULES,
        "dataset_sha256": dataset_sha256,
        "run_id": RUN_ID,
        "base_sha256": sha256_file(base_output),
        "lora_sha256": sha256_file(lora_output),
        "created_at": datetime.now(timezone.utc).isoformat(),
    }
    manifest_path = ANDROID_DIR / "android_model_manifest.json"
    manifest_path.write_text(json.dumps(bundle_manifest, ensure_ascii=False, indent=2), encoding="utf-8")
    zip_path = DRIVE_VERSION_DIR / "zazo_android_model.zip"
    with zipfile.ZipFile(zip_path, "w", compression=zipfile.ZIP_DEFLATED, compresslevel=1) as archive:
        archive.write(base_output, "base.bin")
        archive.write(lora_output, "lora.bin")
        archive.write(manifest_path, "android_model_manifest.json")
    print("تم تجهيز حزمة Android:", zip_path)
    print("ضع zazo_android_model.zip في التطبيق عبر زر تثبيت النموذج المحلي.")
else:
    print("تم تجاوز تجهيز Android. اضبط BUILD_ANDROID_MODEL=True لإنتاج الحزمة.")


## 11. فحص السعة ورفع المحوّل مباشرةً إلى Cloudflare AI

يقرأ هذا المسار ملفات الإصدار من Drive ويرفعها مباشرةً عبر REST API؛ لا يمر عبر حد Worker البالغ 80 MiB. يُنشأ fine-tune جديد لكل إصدار، ولا يُبدّل مؤشر الشات إلا بعد قبول الملفين. يذكر توثيق Cloudflare أن الميزة مجانية خلال البيتا المفتوحة ويتيح اختبار ما يصل إلى 100 محوّل للحساب؛ ويحذّر الدفتر عند 80 ويتوقف عند 95 لإبقاء مساحة احتياطية. حد العدد منفصل عن حصة الاستدلال المجانية اليومية البالغة 10,000 Neurons. لا يحذف هذا الدفتر أي محوّل تلقائيًا؛ احتفظ بنسخ Drive وتحقق منها قبل إزالة إصدار قديم من Cloudflare.


In [ ]:
from requests_toolbelt.multipart.encoder import MultipartEncoder
from safetensors.torch import load_file, save_file
import shutil

MAX_CLOUDFLARE_ADAPTER_BYTES = 300_000_000  # Cloudflare توثق أقل من 300 MB
WORKER_UPLOAD_LIMIT_BYTES = 80 * 1024 * 1024

def list_cloudflare_finetunes():
    payload = cloudflare_json("GET", AI_API_BASE + "/finetunes", CF_AI_TOKEN)
    items = flatten_result_items(payload.get("result"))
    return [item for item in items if isinstance(item, dict) and item.get("id")]

def read_active_lora_pointer():
    text = read_kv_text(ACTIVE_CONTEXT_KV_NAMESPACE_ID, ACTIVE_LORA_KEY)
    if not text:
        return None
    try:
        value = json.loads(text)
    except json.JSONDecodeError:
        return None
    return value if isinstance(value, dict) else None

def show_rotation_candidates(finetunes, active_id):
    ordered = sorted(finetunes, key=lambda item: item.get("created_at", ""), reverse=True)
    protected = {item.get("id") for item in ordered[:KEEP_RECENT_ADAPTERS]}
    if active_id:
        protected.add(active_id)
    candidates = [item for item in reversed(ordered) if item.get("id") not in protected]
    print("أقدم محوّلات غير نشطة يمكن مراجعتها يدويًا:")
    for item in candidates[:10]:
        print(" -", item.get("created_at"), item.get("name"), item.get("id"))
    return candidates

def make_fp16_upload_copy(source_path):
    source_path = Path(source_path)
    if source_path.stat().st_size <= WORKER_UPLOAD_LIMIT_BYTES:
        return source_path
    state = load_file(str(source_path), device="cpu")
    if not any(tensor.is_floating_point() and tensor.dtype == torch.float32 for tensor in state.values()):
        print("الملف أكبر من حد Worker لكنه لا يحوي أوزان FP32 لتخفيضها؛ سنرفعه مباشرةً إلى Cloudflare AI.")
        return source_path
    upload_path = Path("/content") / f"{RUN_ID}-adapter-fp16.safetensors"
    fp16_state = {
        key: tensor.to(torch.float16).contiguous() if tensor.is_floating_point() and tensor.dtype == torch.float32 else tensor.contiguous()
        for key, tensor in state.items()
    }
    save_file(fp16_state, str(upload_path), metadata={"format": "pt"})
    old_size, new_size = source_path.stat().st_size, upload_path.stat().st_size
    print(f"نسخة رفع FP16 مؤقتة: {old_size / (1024**2):.1f} → {new_size / (1024**2):.1f} MiB؛ الأصل في Drive لم يتغير.")
    return upload_path if new_size < old_size else source_path

def upload_finetune_asset(finetune_id, file_name, local_path):
    url = f"{AI_API_BASE}/finetunes/{urllib.parse.quote(finetune_id, safe='')}/finetune-assets/"
    with open(local_path, "rb") as stream:
        multipart = MultipartEncoder(fields={
            "file_name": file_name,
            "file": (file_name, stream, "application/octet-stream"),
        })
        response = requests.post(
            url,
            headers={
                "Authorization": f"Bearer {CF_AI_TOKEN}",
                "Content-Type": multipart.content_type,
            },
            data=multipart,
            timeout=(30, 900),
        )
    try:
        payload = response.json()
    except ValueError as exc:
        raise RuntimeError(f"Cloudflare أعاد استجابة غير JSON لرفع {file_name}: HTTP {response.status_code}") from exc
    if not response.ok or not payload.get("success", False):
        raise RuntimeError(f"فشل رفع {file_name} إلى fine-tune={finetune_id}: {payload}")
    return payload

def create_cloudflare_finetune(name, description):
    payload = cloudflare_json(
        "POST",
        AI_API_BASE + "/finetunes",
        CF_AI_TOKEN,
        json={"model": CF_AI_MODEL, "name": name, "description": description},
    )
    result = flatten_result_items(payload.get("result"))
    created = next((item for item in result if isinstance(item, dict) and item.get("id")), None)
    if not created:
        raise RuntimeError(f"Cloudflare لم يُرجع معرف fine-tune: {payload}")
    return created

# تحقق من صلاحية config ووزن المحوّل قبل إنشاء مورد Cloudflare.
if adapter_config.get("model_type") != "gemma":
    raise RuntimeError("adapter_config.json يجب أن يحتوي model_type=gemma.")
if not isinstance(adapter_config.get("r"), int) or adapter_config["r"] > 32:
    raise RuntimeError(f"رتبة LoRA غير مدعومة: r={adapter_config.get('r')}؛ الحد الأعلى المعلن 32.")

upload_weights_path = make_fp16_upload_copy(adapter_weights_path)
weight_bytes = upload_weights_path.stat().st_size
if weight_bytes >= MAX_CLOUDFLARE_ADAPTER_BYTES:
    raise RuntimeError(
        f"حجم المحوّل بعد محاولة FP16 هو {weight_bytes / (1024**2):.1f} MiB ويتجاوز حد Cloudflare (<300 MB). "
        "أعد التدريب برتبة LORA_R=4 وtarget_modules أقل مثل ['q_proj', 'v_proj']."
    )

existing_finetunes = list_cloudflare_finetunes()
active_pointer_before = read_active_lora_pointer()
active_id_before = active_pointer_before.get("id") if active_pointer_before else None
print(f"عدد محوّلات Cloudflare الحالية: {len(existing_finetunes)} / 100")
if len(existing_finetunes) >= WARN_AT_ADAPTERS:
    print(f"تحذير تدوير: وصل العدد إلى {len(existing_finetunes)}؛ راجع النسخ القديمة المؤرشفة في Drive.")
    rotation_candidates = show_rotation_candidates(existing_finetunes, active_id_before)
if len(existing_finetunes) >= STOP_AT_ADAPTERS:
    raise RuntimeError(
        f"أوقفت إنشاء fine-tune عند {len(existing_finetunes)} محوّلًا لإبقاء خمس خانات احتياطية. "
        "أرشف نسخة قديمة وتحقق منها ثم أزلها من Cloudflare يدويًا، أو راجع سياسة الحساب قبل المتابعة. لا حذف تلقائيًا."
    )

finetune_name = f"zazo-r{LORA_R}-{RUN_TIMESTAMP.replace(':', '').replace('-', '')}"
created_finetune = create_cloudflare_finetune(
    finetune_name,
    f"Zazo adapter; data_sha256={dataset_sha256[:16]}; source={manifest_key}",
)
new_finetune_id = created_finetune["id"]
print("أنشئ fine-tune:", finetune_name, new_finetune_id)
try:
    upload_finetune_asset(new_finetune_id, "adapter_model.safetensors", upload_weights_path)
    upload_finetune_asset(new_finetune_id, "adapter_config.json", adapter_config_path)
except Exception as exc:
    training_metadata["cloudflare_status"] = "upload_failed"
    training_metadata["fine_tune_id"] = new_finetune_id
    training_metadata["upload_error"] = str(exc)[:1000]
    (DRIVE_VERSION_DIR / "training_manifest.json").write_text(json.dumps(training_metadata, ensure_ascii=False, indent=2), encoding="utf-8")
    raise RuntimeError(
        f"فشل رفع المحوّل؛ لم يتغير مؤشر الشات وسيستمر الإصدار السابق. fineTuneId={new_finetune_id}. "
        "احتفظ بهذا المعرّف للتنظيف أو استكمال التحقق."
    ) from exc

# لا نغيّر الإصدار النشط إلا بعد قبول Cloudflare لكلا الملفين.
active_pointer = {
    "id": new_finetune_id,
    "name": finetune_name,
    "model": CF_AI_MODEL,
    "createdAt": datetime.now(timezone.utc).isoformat(),
    "sourceManifestKey": manifest_key,
    "datasetSha256": dataset_sha256,
    "archivePath": str(DRIVE_VERSION_DIR),
}
write_kv_json(ACTIVE_CONTEXT_KV_NAMESPACE_ID, ACTIVE_LORA_KEY, active_pointer)
readback = None
for _ in range(6):
    readback = read_active_lora_pointer()
    if readback and readback.get("id") == new_finetune_id:
        break
    time.sleep(2)
if not readback or readback.get("id") != new_finetune_id:
    raise RuntimeError(
        f"تم رفع المحوّل لكن تعذر التحقق من مؤشر KV بعد إعادة المحاولة. المعرف الجديد: {new_finetune_id}"
    )

training_metadata.update({
    "status": "active",
    "fine_tune_id": new_finetune_id,
    "cloudflare_model": CF_AI_MODEL,
    "active_pointer_key": ACTIVE_LORA_KEY,
    "uploaded_weights_bytes": weight_bytes,
})
(DRIVE_VERSION_DIR / "training_manifest.json").write_text(
    json.dumps(training_metadata, ensure_ascii=False, indent=2), encoding="utf-8"
)
latest_training_path.write_text(json.dumps(training_metadata, ensure_ascii=False, indent=2), encoding="utf-8")
print("نجح الرفع والتفعيل.")
print("معرف LoRA النشط:", new_finetune_id)
print("نسخة Drive المؤرشفة:", DRIVE_VERSION_DIR)
print("ملاحظة: قد يستغرق انتشار KV وقتًا قصيرًا؛ إذا فشل الرفع فلن يتغير الإصدار النشط.")

## 12. تجربة محلية اختيارية بعد التدريب

هذه الخلية تختبر المحوّل داخل Colab فقط، ولا تنشره أو تغير مؤشر الإنتاج.

In [ ]:
model.eval()
test_messages = [{"role": "user", "content": "كيف يربط القرآن بين التدبر والوعي؟ أجب مع ذكر مرجع الآية."}]
prompt = tokenizer.apply_chat_template(test_messages, tokenize=False, add_generation_prompt=True)
inputs = tokenizer(prompt, return_tensors="pt").to(model.device)
with torch.no_grad():
    output = model.generate(**inputs, max_new_tokens=160, do_sample=True, temperature=0.7, top_p=0.9)
print(tokenizer.decode(output[0][inputs["input_ids"].shape[1]:], skip_special_tokens=True))


## ملاحظات تشغيل وأمان

- شغّل GitHub Action `rag-dataset-prep` قبل التدريب إذا كانت المواد الجديدة لم تدخل في أحدث manifest. الدفتر يعرض وقت التصدير وبصمة المحتوى وعدد `readyRows`، ويتوقف إذا كانت البيانات قديمة أو لم يتطابق العدد.
- التدريب يقرأ `readyChunkKeys` فقط؛ بيانات `reviewChunkKeys` لا تدخل التدريب.
- Cloudflare KV يحفظ قيمة `active-lora` الصغيرة فقط، لا ملفات الأوزان. ملفات كل إصدار تبقى في مجلد مستقل على Google Drive، وCloudflare AI يحتفظ بالمحوّلات المرفوعة.
- كل إصدار ناجح ينشئ fine-tune جديدًا. التحويل إلى FP16 يتم على نسخة رفع مؤقتة فقط إذا تجاوز الملف 80 MiB؛ النسخة الأصلية في Drive لا تتغير. إذا تجاوز الملف بعد ذلك 300 MB، أعد التدريب برتبة أقل أو وحدات مستهدفة أقل.
- العدد يُفحص قبل الإنشاء: تحذير عند 80 وإيقاف عند 95 للحفاظ على خمس خانات احتياطية. هذا حد عدد المحوّلات، وهو منفصل عن حصة Workers AI المجانية البالغة 10,000 Neurons يوميًا. لا يحذف الدفتر محوّلات قديمة تلقائيًا؛ احتفظ بالنسخة النشطة وآخر خمسة في Cloudflare، وأرشف أوزان أي إصدار تريد إزالته في Drive أولًا.
- أسرار Cloudflare وHugging Face تبقى في Colab Secrets ولا تُكتب في الدفتر أو Drive أو Git. لا تشارك نسخة دفتر أعطيتها وصولًا إلى Secrets.
- قد يطلب Colab تفويض Google Drive عند بدء runtime جديد؛ لا تخزّن OAuth token داخل الدفتر.
- يجب قبول شروط Gemma على Hugging Face قبل تنزيله. يلزم GPU فعليًا؛ CPU وTPU لا يعملان مع هذا الإعداد.
- راجع جودة البيانات والأجوبة علميًا قبل التدريب. هذا fine-tuning لا يعوّض مراجعة المحتوى.